In [ ]:
from pymongo import MongoClient
import os
from pymongo import MongoClient
from dotenv import  load_dotenv
import json
from langchain_text_splitters import RecursiveJsonSplitter
load_dotenv()
mongo_uri = os.getenv('MONGO_URI')
client = MongoClient(mongo_uri)
db = client['cv_platform']
candidates = db['candidates']


In [3]:
yasmine = candidates.find_one({'name': 'Yasmine Goubantini'},{'_id':0,'versions.raw_text': 0})


In [ ]:
versions=yasmine["versions"]
print(len(versions))


In [6]:
splitter=RecursiveJsonSplitter(max_chunk_size=10,min_chunk_size=5)
chunked=[]
for v in versions:
    yasmine_chunked=splitter.split_json(v)
    chunked.append(yasmine_chunked)
for chunk in chunked:
    print('==new version==\n')
    for c in chunk:
        print(c)
        print('\n')
    

print(len(yasmine_chunked))

==new version==

{'version_number': 1}


{'structured': {'name': 'Yasmine Goubantini'}}


{'structured': {'summary': 'Yasmine Goubantini is a Transformation Technology  Consultant with over 2 years of experience in digital and data consulting. She combines an academic background in Data Science with consulting and project management expertise to help organizations leverage technology & data for strategic and operational excellence. Certified in Microsoft Power BI (PL-300) and pursuing the CDMP – Associate (DAMA International) certification, she applies structured methodologies and analytical thinking to deliver actionable insights, improve data-driven decision-making, and support innovation.'}}


{'structured': {'expertise_areas': [{'category': 'Digital Solutions Delivery', 'description': 'Assisting in the implementation of IT and data initiatives by aligning business objectives with technical design and operational performance.'}, {'category': 'Digital Transformation Strategy', 'descr

In [42]:
def candidats_no_projects_no_expirience() : 
    return candidates.find(
    {
        "$and": [
            {
                "versions": {
                    "$elemMatch": {
                        "structured.experience": []
                    }
                }
            },
            {
                "versions": {
                    "$elemMatch": {
                        "structured.projects": []
                    }
                }
            }
        ]
    },
    {
        "_id": 0,
        
    }
)




In [44]:
candidates_empty_exp_or_projects=candidats_no_projects_no_expirience()
for c in candidates_empty_exp_or_projects:
    print(c["name"])
    
    for v in c["versions"]:
        experience = v.get("structured", {}).get("experience", [])
        projects = v.get("structured", {}).get("projects", [])

        if experience == [] and projects == []:
            print("Version:", v["version_number"])